In [13]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-6ded8d6b-03c8-49f6-7855-ba633f3068ff, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [14]:
GITHUB_USERNAME = "Koreshzy0"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: Koreshzy0


In [15]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import re
import time
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants for 2D shared-memory tiling (16x16 interior + 1-cell halo apron)
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
REPORT_EVERY = 100
DT_SAFETY = 0.5  # dt = DT_SAFETY * dt_max

DIRS = ("up", "down", "left", "right")
OPP = {"up": "down", "down": "up", "left": "right", "right": "left"}
TAG = {"up": 0, "down": 1, "left": 2, "right": 3}


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf8")).hexdigest()
    return int(digest[:16], 16)


def allreduce_scalar(cart, value, op):
    s = np.array([value], dtype=np.float64)
    r = np.empty(1, dtype=np.float64)
    cart.Allreduce(s, r, op=op)
    return float(r[0])


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    cart = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    rank = cart.Get_rank()
    row_rank, col_rank = cart.Get_coords(rank)
    nbr_up, nbr_down = cart.Shift(0, 1)      # axis 0: (source=up, dest=down)
    nbr_left, nbr_right = cart.Shift(1, 1)   # axis 1: (source=left, dest=right)

    # Bind process to GPU 0 on Google Colab
    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = cart.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    return {
        "cart": cart,
        "rank": rank,
        "size": size,
        "row_rank": row_rank,
        "col_rank": col_rank,
        "nbr": {"up": nbr_up, "down": nbr_down, "left": nbr_left, "right": nbr_right},
        "gpu_uuid": gpu_uuid,
        "seed": instance_seed,
        "github_user": github_user.strip(),
        "ny": GLOBAL_NY // 2,
        "nx": GLOBAL_NX // 2,
    }


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ==============================================================================
def initialize_subdomains(ctx):
    ny, nx = ctx["ny"], ctx["nx"]
    dx = 1.0 / (GLOBAL_NX + 1)
    # global node indices 0..513 (0 and 513 are the physical Dirichlet boundary)
    gi = ctx["row_rank"] * ny + np.arange(ny + 2)
    gj = ctx["col_rank"] * nx + np.arange(nx + 2)
    X, Y = np.meshgrid(gj * dx, gi * dx)  # X[i,j] = x[j], Y[i,j] = y[i]

    # heterogeneous diffusivity in [0.08, 0.24]
    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * X) * np.cos(2.0 * np.pi * Y)

    # seed-derived pulses (every rank draws the same sequence -> consistent global field)
    rng = np.random.default_rng(ctx["seed"])
    n_p = 3
    cx = rng.uniform(0.25, 0.75, n_p)
    cy = rng.uniform(0.25, 0.75, n_p)
    amp = rng.uniform(0.5, 1.0, n_p)
    sig = rng.uniform(0.05, 0.10, n_p)
    sx, sy = rng.uniform(0.3, 0.7, 2)

    u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    for p in range(n_p):
        u += amp[p] * np.exp(-((X - cx[p]) ** 2 + (Y - cy[p]) ** 2) / (2.0 * sig[p] ** 2))

    # small localized source (kept small so damping dominates -> energy decays)
    S = 1.0e-3 * np.exp(-((X - sx) ** 2 + (Y - sy) ** 2) / (2.0 * 0.04 ** 2))

    nbr = ctx["nbr"]
    if nbr["up"] == MPI.PROC_NULL:
        u[0, :] = 0.0
    if nbr["down"] == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0
    if nbr["left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if nbr["right"] == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return (np.ascontiguousarray(u), np.ascontiguousarray(kappa), np.ascontiguousarray(S))

Overwriting distributed_cuda_solver.py


In [16]:
%%writefile -a distributed_cuda_solver.py


# ==============================================================================
# TASK 3: Halo pack/unpack kernels + pinned asynchronous exchange
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, s_up, s_down, s_left, s_right, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        s_up[k] = u[1, k + 1]        # row 1 (contiguous)
        s_down[k] = u[ny, k + 1]     # row ny
    if k < ny:
        s_left[k] = u[k + 1, 1]      # column 1 (strided)
        s_right[k] = u[k + 1, nx]    # column nx


@cuda.jit
def unpack_halos_kernel(u, r_up, r_down, r_left, r_right, ny, nx):
    k = cuda.grid(1)
    if k < nx:
        u[0, k + 1] = r_up[k]
        u[ny + 1, k + 1] = r_down[k]
    if k < ny:
        u[k + 1, 0] = r_left[k]
        u[k + 1, nx + 1] = r_right[k]


class HaloBuffers:
    def __init__(self, ny, nx):
        n = {"up": nx, "down": nx, "left": ny, "right": ny}
        self.d_send = {d: cuda.device_array(n[d], dtype=np.float64) for d in DIRS}
        self.d_recv = {d: cuda.device_array(n[d], dtype=np.float64) for d in DIRS}
        self.h_send = {d: cuda.pinned_array(n[d], dtype=np.float64) for d in DIRS}
        self.h_recv = {d: cuda.pinned_array(n[d], dtype=np.float64) for d in DIRS}
        for d in DIRS:
            self.h_send[d][:] = 0.0
            self.h_recv[d][:] = 0.0  # stays 0 for MPI.PROC_NULL neighbours -> Dirichlet u=0
        self.threads = 128
        self.blocks = (max(ny, nx) + self.threads - 1) // self.threads


def exchange_halos_async(ctx, hb, d_u, stream):
    """Stage A (pack + async D2H on `stream`) and first half of Stage B (post Irecv/Isend).
    Returns the list of MPI requests; call finish_halo_exchange() to complete."""
    ny, nx = ctx["ny"], ctx["nx"]
    cart, nbr = ctx["cart"], ctx["nbr"]

    pack_halos_kernel[hb.blocks, hb.threads, stream](
        d_u, hb.d_send["up"], hb.d_send["down"], hb.d_send["left"], hb.d_send["right"], ny, nx
    )
    for d in DIRS:
        hb.d_send[d].copy_to_host(hb.h_send[d], stream=stream)  # async D2H into pinned memory
    stream.synchronize()  # host send buffers are now valid

    reqs = []
    for d in DIRS:  # neighbour in direction d sent us the strip it tagged TAG[OPP[d]]
        reqs.append(cart.Irecv(hb.h_recv[d], source=nbr[d], tag=TAG[OPP[d]]))
    for d in DIRS:
        reqs.append(cart.Isend(hb.h_send[d], dest=nbr[d], tag=TAG[d]))
    return reqs


def finish_halo_exchange(ctx, hb, d_u, reqs, stream):
    """Stage C (first part): Waitall, async H2D, unpack into ghost cells."""
    MPI.Request.Waitall(reqs)
    for d in DIRS:
        hb.d_recv[d].copy_to_device(hb.h_recv[d], stream=stream)
    unpack_halos_kernel[hb.blocks, hb.threads, stream](
        d_u, hb.d_recv["up"], hb.d_recv["down"], hb.d_recv["left"], hb.d_recv["right"],
        ctx["ny"], ctx["nx"],
    )


# ==============================================================================
# TASK 4: Shared-memory tiled stencil + dual-stream orchestrator
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u_in, u_out, kappa, src, r_start, r_end, c_start, c_end,
                         dt, inv_dx2, inv_dy2, lam):
    smem = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty

    in_i = i <= r_end + 1
    in_j = j <= c_end + 1
    if in_i and in_j:
        smem[ty + 1, tx + 1] = u_in[i, j]              # center
        if ty == 0:
            smem[0, tx + 1] = u_in[i - 1, j]           # top apron
        if tx == 0:
            smem[ty + 1, 0] = u_in[i, j - 1]           # left apron
    if ty == TILE_DIM - 1 and (i + 1 <= r_end + 1) and in_j:
        smem[TILE_DIM + 1, tx + 1] = u_in[i + 1, j]    # bottom apron
    if tx == TILE_DIM - 1 and (j + 1 <= c_end + 1) and in_i:
        smem[ty + 1, TILE_DIM + 1] = u_in[i, j + 1]    # right apron
    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        c = smem[ty + 1, tx + 1]
        n = smem[ty, tx + 1]
        s = smem[ty + 2, tx + 1]
        w = smem[ty + 1, tx]
        e = smem[ty + 1, tx + 2]
        k0 = kappa[i, j]
        k_n = 0.5 * (k0 + kappa[i - 1, j])
        k_s = 0.5 * (k0 + kappa[i + 1, j])
        k_w = 0.5 * (k0 + kappa[i, j - 1])
        k_e = 0.5 * (k0 + kappa[i, j + 1])
        rhs = ((k_s * (s - c) - k_n * (c - n)) * inv_dy2
               + (k_e * (e - c) - k_w * (c - w)) * inv_dx2
               - lam * c * c * c + src[i, j])
        u_out[i, j] = c + dt * rhs


def launch_stencil(stream, d_in, d_out, d_k, d_S, box, dt, inv_dx2, inv_dy2):
    r0, r1, c0, c1 = box
    gy = (r1 - r0 + 1 + TILE_DIM - 1) // TILE_DIM
    gx = (c1 - c0 + 1 + TILE_DIM - 1) // TILE_DIM
    tiled_stencil_kernel[(gx, gy), (TILE_DIM, TILE_DIM), stream](
        d_in, d_out, d_k, d_S, r0, r1, c0, c1, dt, inv_dx2, inv_dy2, LAMBDA_DAMP
    )


class SolverState:
    def __init__(self, ctx, u0, kappa, S, dt, dx):
        self.ctx = ctx
        ny, nx = ctx["ny"], ctx["nx"]
        self.ny, self.nx = ny, nx
        self.d_u = cuda.to_device(u0)
        self.d_un = cuda.to_device(u0.copy())
        self.d_k = cuda.to_device(kappa)
        self.d_S = cuda.to_device(S)
        self.hb = HaloBuffers(ny, nx)
        self.s_int = cuda.stream()
        self.s_bnd = cuda.stream()
        self.dt = dt
        self.inv_dx2 = 1.0 / (dx * dx)
        self.inv_dy2 = 1.0 / (dx * dx)
        self.interior_box = (2, ny - 1, 2, nx - 1)
        self.full_box = (1, ny, 1, nx)
        self.ring_boxes = [(1, 1, 1, nx), (ny, ny, 1, nx), (2, ny - 1, 1, 1), (2, ny - 1, nx, nx)]

    def swap(self):
        self.d_u, self.d_un = self.d_un, self.d_u


def step_serialized(S):
    reqs = exchange_halos_async(S.ctx, S.hb, S.d_u, S.s_bnd)
    finish_halo_exchange(S.ctx, S.hb, S.d_u, reqs, S.s_bnd)
    launch_stencil(S.s_bnd, S.d_u, S.d_un, S.d_k, S.d_S, S.full_box, S.dt, S.inv_dx2, S.inv_dy2)
    S.s_bnd.synchronize()
    S.swap()


def step_overlapped(S):
    # Stage A + B(post): pack, D2H, sync, Irecv/Isend on stream_boundary
    reqs = exchange_halos_async(S.ctx, S.hb, S.d_u, S.s_bnd)
    # Stage B: deep interior on stream_interior while MPI is in flight
    launch_stencil(S.s_int, S.d_u, S.d_un, S.d_k, S.d_S, S.interior_box, S.dt, S.inv_dx2, S.inv_dy2)
    # Stage C: wait, H2D, unpack, then 4 boundary rings on stream_boundary
    finish_halo_exchange(S.ctx, S.hb, S.d_u, reqs, S.s_bnd)
    for box in S.ring_boxes:
        launch_stencil(S.s_bnd, S.d_u, S.d_un, S.d_k, S.d_S, box, S.dt, S.inv_dx2, S.inv_dy2)
    S.s_int.synchronize()
    S.s_bnd.synchronize()
    S.swap()

Appending to distributed_cuda_solver.py


In [17]:
%%writefile -a distributed_cuda_solver.py


# ==============================================================================
# TASK 5: reductions, benchmark, PTX audit, manifest
# ==============================================================================
def run_mode(ctx, mode, steps, u0, kappa, S0, dt, dx, verbose=True):
    cart = ctx["cart"]
    ny, nx = ctx["ny"], ctx["nx"]
    st = SolverState(ctx, u0, kappa, S0, dt, dx)
    step_fn = step_serialized if mode == "serialized" else step_overlapped
    dxdy = dx * dx

    def local_energy(u_host):
        return float(u_host[1:ny + 1, 1:nx + 1].sum() * dxdy)

    trace = []
    e0 = allreduce_scalar(cart, local_energy(st.d_u.copy_to_host()), MPI.SUM)
    trace.append({"step": 0, "energy": e0, "l2_residual": 0.0})

    wall_s = 0.0
    event_ms = 0.0
    ev_a, ev_b = cuda.event(), cuda.event()
    cart.Barrier()
    for s in range(1, steps + 1):
        report = (s % REPORT_EVERY == 0)
        old = st.d_u.copy_to_host() if report else None
        t0 = time.perf_counter()
        ev_a.record(stream=st.s_bnd)
        step_fn(st)
        ev_b.record(stream=st.s_bnd)
        ev_b.synchronize()
        wall_s += time.perf_counter() - t0
        event_ms += cuda.event_elapsed_time(ev_a, ev_b)
        if report:
            new = st.d_u.copy_to_host()
            e = allreduce_scalar(cart, local_energy(new), MPI.SUM)
            diff = new[1:ny + 1, 1:nx + 1] - old[1:ny + 1, 1:nx + 1]
            l2 = math.sqrt(allreduce_scalar(cart, float((diff * diff).sum()), MPI.SUM))
            trace.append({"step": s, "energy": e, "l2_residual": l2})
    cart.Barrier()

    wall_ms = allreduce_scalar(cart, wall_s * 1e3, MPI.MAX)
    ev_ms = allreduce_scalar(cart, event_ms, MPI.MAX)
    return {"u": st.d_u.copy_to_host(), "trace": trace, "wall_ms": wall_ms, "event_ms": ev_ms}


def ptx_audit():
    asm = tiled_stencil_kernel.inspect_asm()
    if isinstance(asm, dict):
        asm = "\n".join(asm.values())
    counts = {
        "st.shared.f64": len(re.findall(r"\bst\.shared\.f64\b", asm)),
        "ld.shared.f64": len(re.findall(r"\bld\.shared\.f64\b", asm)),
        "bar.sync": len(re.findall(r"\bbar\.sync\b", asm)),
    }
    return counts, all(v > 0 for v in counts.values())


def run_solver(github_user: str):
    ctx = init_distributed_context(github_user)
    cart, rank = ctx["cart"], ctx["rank"]
    u0, kappa, S0 = initialize_subdomains(ctx)

    dx = 1.0 / (GLOBAL_NX + 1)
    kappa_max = allreduce_scalar(cart, float(kappa.max()), MPI.MAX)
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = DT_SAFETY * dt_max

    if rank == 0:
        print(f"GITHUB_USERNAME     : {ctx['github_user']}")
        print(f"GPU UUID            : {ctx['gpu_uuid']}")
        print(f"INSTANCE_SEED       : {ctx['seed']}")
        print(f"MPI ranks / grid    : {ctx['size']} / 2x2 Cartesian (non-periodic)")
        print(f"kappa_max={kappa_max:.6f}  dx={dx:.8e}  dt_max={dt_max:.6e}  dt={dt:.6e}")
    for r in range(4):
        cart.Barrier()
        if r == rank:
            print(f"[rank {rank}] coords=({ctx['row_rank']},{ctx['col_rank']}) nbrs={ctx['nbr']}")
    cart.Barrier()

    # JIT warm-up (excluded from timings)
    run_mode(ctx, "serialized", 2, u0, kappa, S0, dt, dx)
    run_mode(ctx, "overlapped", 2, u0, kappa, S0, dt, dx)

    ser = run_mode(ctx, "serialized", STEPS, u0, kappa, S0, dt, dx)
    ovl = run_mode(ctx, "overlapped", STEPS, u0, kappa, S0, dt, dx)

    ny, nx = ctx["ny"], ctx["nx"]
    local_diff = float(np.abs(ser["u"][1:ny + 1, 1:nx + 1] - ovl["u"][1:ny + 1, 1:nx + 1]).max())
    parity = allreduce_scalar(cart, local_diff, MPI.MAX)

    counts, ptx_ok = ptx_audit()

    energies = [t["energy"] for t in ovl["trace"]]
    monotone = all(energies[k + 1] <= energies[k] for k in range(len(energies) - 1))

    if rank == 0:
        print("\n--- Allreduce energy / L2 residual trace (overlapped) ---")
        for t in ovl["trace"]:
            print(f"step {t['step']:4d}  E = {t['energy']:.10e}  ||du||_2 = {t['l2_residual']:.6e}")
        print(f"monotone energy decay: {monotone}")
        print("\n--- Serialized trace ---")
        for t in ser["trace"]:
            print(f"step {t['step']:4d}  E = {t['energy']:.10e}  ||du||_2 = {t['l2_residual']:.6e}")
        print(f"\nserialized_ms = {ser['wall_ms']:.3f} (CUDA events: {ser['event_ms']:.3f})")
        print(f"overlapped_ms = {ovl['wall_ms']:.3f} (CUDA events: {ovl['event_ms']:.3f})")
        print(f"speedup (serialized/overlapped) = {ser['wall_ms'] / ovl['wall_ms']:.3f}x")
        print(f"parity_max_diff = {parity:.3e}  (required < 1e-12)")
        print(f"PTX counts: {counts}  ptx_audit_passed={ptx_ok}")

        manifest = {
            "GITHUB_USERNAME": ctx["github_user"],
            "gpu_uuid": ctx["gpu_uuid"],
            "instance_seed": ctx["seed"],
            "mpi_ranks": ctx["size"],
            "cartesian_dims": [2, 2],
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [ny + 2, nx + 2],
            "steps": STEPS,
            "kappa_max": kappa_max,
            "dx": dx,
            "dt_max": dt_max,
            "dt": dt,
            "serialized_ms": ser["wall_ms"],
            "overlapped_ms": ovl["wall_ms"],
            "serialized_cuda_event_ms": ser["event_ms"],
            "overlapped_cuda_event_ms": ovl["event_ms"],
            "parity_max_diff": parity,
            "ptx_counts": counts,
            "ptx_audit_passed": bool(ptx_ok),
            "energy_trace": ovl["trace"],
            "energy_monotone_decreasing": bool(monotone),
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("\nWrote submission_manifest.json")

    if parity >= 1e-12 or not ptx_ok:
        raise RuntimeError("Verification failed (parity or PTX audit).")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        raise SystemExit("usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    run_solver(sys.argv[1])

Appending to distributed_cuda_solver.py


In [18]:
!wc -l distributed_cuda_solver.py
!tail -n 3 distributed_cuda_solver.py

439 distributed_cuda_solver.py
    if len(sys.argv) < 2:
        raise SystemExit("usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    run_solver(sys.argv[1])


In [19]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

GITHUB_USERNAME     : Koreshzy0
GPU UUID            : GPU-6ded8d6b-03c8-49f6-7855-ba633f3068ff
INSTANCE_SEED       : 8694547779874023558
MPI ranks / grid    : 4 / 2x2 Cartesian (non-periodic)
kappa_max=0.240000  dx=1.94931774e-03  dt_max=3.958172e-06  dt=1.979086e-06
[rank 0] coords=(0,0) nbrs={'up': -2, 'down': 2, 'left': -2, 'right': 1}
[rank 1] coords=(0,1) nbrs={'up': -2, 'down': 3, 'left': 0, 'right': -2}
[rank 2] coords=(1,0) nbrs={'up': 0, 'down': -2, 'left': -2, 'right': 3}
[rank 3] coords=(1,1) nbrs={'up': 1, 'down': -2, 'left': 2, 'right': -2}

--- Allreduce energy / L2 residual trace (overlapped) ---
step    0  E = 5.9522844497e-02  ||du||_2 = 0.000000e+00
step  100  E = 5.9522485845e-02  ||du||_2 = 7.997854e-03
step  200  E = 5.9522137641e-02  ||du||_2 = 7.791019e-03
step  300  E = 5.9521797315e-02  ||du||_2 = 7.592168e-03
step  400  E = 5.9521464130e-02  ||du||_2 = 7.400879e-03
monotone energy decay: True

--- Serialized trace ---
step    0  E = 5.9522844497e-02  ||du||_2 

### Analytical & Forensic Engineering Responses

- **Question 1 (Memory Coalescing & Strided Column Packing):** The local array is C-contiguous with a row pitch of 258 doubles (2064 B). A row strip `u[1, 1:nx+1]` is 256 consecutive doubles, but the column strips `u[1:ny+1, 1]` and `u[1:ny+1, nx]` have their elements 2064 B apart. At warp level, 32 threads reading one column touch 32 different 32-byte sectors (32 memory transactions, ~1024 B moved for 256 B of useful data, 25% efficiency), whereas a contiguous row needs only 8 sectors. A single DMA (`cudaMemcpy`) can only move one contiguous byte range (or a pitched 2D copy), not an arbitrary strided gather, so a column cannot be sent in one contiguous DMA. If I copied the contiguous bounding span instead, for one column the span from `u[1,c]` to `u[ny,c]` is (ny-1)*258 + 1 = 65,791 doubles = 526,328 B versus the 256*8 = 2,048 B actually needed. For both columns that is 1,052,656 B transferred instead of 4,096 B, i.e. about **1,048,560 extra bytes (~1 MiB) per rank per step** (~257x overhead). `pack_halos_kernel` gathers each column on the GPU into a contiguous 2,048 B buffer, so PCIe moves only the 4 strips x 2,048 B = 8,192 B per rank per step.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):** Thread (tx=0, ty=0): writes `smem[1,1]` (center), `smem[0,1]` (top apron, since ty==0) and `smem[1,0]` (left apron, since tx==0); reads `smem[1,1]`, `smem[0,1]`, `smem[2,1]`, `smem[1,0]`, `smem[1,2]`. Thread (tx=15, ty=15): writes `smem[16,16]` (center), `smem[17,16]` (bottom apron, ty==15) and `smem[16,17]` (right apron, tx==15); reads `smem[16,16]`, `smem[15,16]`, `smem[17,16]`, `smem[16,15]`, `smem[16,17]`. On CC 7.5 (32 banks x 4 B) an f64 occupies two banks and a 64-bit access is serviced per half-warp (16 threads x 8 B = 128 B). In a 16x16 block a half-warp is one `ty` row, so the threads differ in `tx`. Along a row the 16 addresses are consecutive doubles (128 B), covering all 32 banks exactly once, so there is **no bank conflict**. Down a column the stride is 18 doubles = 36 words, so the bank is (36*ty) mod 32 = (4*ty) mod 32 and rows ty and ty+8 map to the same banks: a column access by 16 threads would give a **2-way bank conflict**. My kernel never accesses a column with a whole half-warp (north/south neighbours are still read as `smem[ty, tx+1]` / `smem[ty+2, tx+1]` across tx; the only column-wise accesses are the tx==0 left-apron writes, at most 2 threads per warp with banks 4 apart), so the stencil is conflict-free.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):** `tiled_stencil_kernel.inspect_asm()` in my run reported `st.shared.f64` = **5**, `ld.shared.f64` = **5**, `bar.sync` = **1** (`ptx_audit_passed = True`). The 5 stores are the center plus the four apron loads (top, left, bottom, right); the 5 loads are the center and its N/S/W/E neighbours; the single `bar.sync` is `cuda.syncthreads()` between loading and stencil evaluation. A 16x16 block is 8 warps (warp w holds rows ty = 2w and 2w+1). A thread reads shared cells written by threads of other warps (for example ty=2, in warp 1, reads its north neighbour `smem[2,.]`, which is the center written by ty=1 in warp 0; aprons are written by only a few threads). Without `cuda.syncthreads()` this is a read-after-write (RAW) hazard: warps run independently, so a warp can read cells that have not been written yet, getting uninitialized or stale shared memory from a previous block. The result would be nondeterministic and would break the serialized-vs-overlapped parity.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):** `mpirun -np 4` creates four OS processes, each with its own CUDA context on the same Tesla T4. Without MPS the driver time-slices the contexts: kernels from different processes are not executed concurrently on the SMs but interleaved by context switches, and the copy (DMA) engines and the PCIe link are shared and serialized between processes as well. Measured (max over ranks, 400 steps): `serialized_ms` = **4093.676** (CUDA events 4009.590) and `overlapped_ms` = **4376.438** (CUDA events 4120.886), i.e. a speedup of **0.935x** (overlap about 6.9% slower; 10.23 ms vs 10.94 ms per step). Absolute timings vary between runs on a shared Colab T4 (an earlier run of the same code gave 2080.694 ms vs 2216.789 ms, speedup 0.939x), but the ratio is stable at about 0.94x, so overlap does not pay off in either run. Overlap cannot help here for two reasons: (1) there is little communication to hide, since the halo messages are only 2,048 B each and travel through shared memory between processes on one node, and (2) the interior kernel of one rank cannot truly run in parallel with the other ranks' kernels or copies because the SMs and DMA engines are time-sliced across the four contexts. Meanwhile the overlapped mode pays extra costs: 5 kernel launches per step (1 interior + 4 boundary rings) instead of 1, and two stream synchronizations. On four dedicated GPUs over InfiniBand each rank owns its own SMs and DMA engines and the network latency is much larger, so the interior compute hides real communication time and the overlap gain is positive.

- **Question 5 (Point-to-Point Message Deadlock & MPI.PROC_NULL Semantics):** A blocking `Send` returns immediately only if MPI can buffer the message (eager protocol, for messages under the eager threshold, typically a few KB up to tens of KB depending on the transport); my 2,048 B halos are below that, so this would usually appear to work. On large grids the message exceeds the threshold and MPI switches to the rendezvous protocol, where `Send` blocks until the matching `Recv` is posted. If every rank calls `Send` first in the same fixed direction (e.g. up), each rank waits for its neighbour to post a `Recv`, but that neighbour is itself stuck in `Send`: a circular wait, hence deadlock. `Isend/Irecv` + `Waitall` avoids it, since all receives are posted before any wait. `MPI.PROC_NULL` turns a send or receive with a non-existent neighbour into an immediate no-op (a receive leaves the buffer untouched). Rank (0,0) has `up` and `left` equal to PROC_NULL (my output: `{'up': -2, 'down': 2, 'left': -2, 'right': 1}`), so it runs exactly the same code path as interior ranks without any `if` branches, and its zero-initialized receive buffers are unpacked as u = 0.0, which is the Dirichlet condition.

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):** From my run: kappa_max = **0.240000**, dx = 1/513 = 1.94931774e-3, dx^2 = 3.7998e-6, so dt_max = dx^2 / (4*kappa_max) = **3.958172e-6**. The simulation used dt = 0.5 * dt_max = **1.979086e-6** (ratio 0.5 < 1, so the scheme is within the explicit stability limit). Allreduce energy trace (E = sum u * dx * dy): step 0: 5.9522844497e-02; step 100: 5.9522485845e-02; step 200: 5.9522137641e-02; step 300: 5.9521797315e-02; step 400: 5.9521464130e-02. Every value is smaller than the previous one (`monotone energy decay: True`); the decrements per 100 steps are 3.587e-07, 3.482e-07, 3.403e-07 and 3.332e-07, a total drop of 1.38e-06 (about 2.3e-5 relative), and the L2 residual also decreases (7.998e-03, 7.791e-03, 7.592e-03, 7.401e-03). This is dissipative behaviour, not strict conservation: the zero Dirichlet edges let diffusion flux leave the domain, the -lambda*u^3 damping removes energy, and the small source term (amplitude 1e-3) adds much less than is removed. There is no blow-up or oscillation, which confirms stability.